# 🔄 Osnovni radni tokovi agenata s Microsoft Foundry (Python)

## 📋 Vodič za orkestraciju radnih tokova

Ovaj bilježnik uvodi moćne mogućnosti **Workflow Buildera** u Microsoft Agent Frameworku. Naučite kako stvoriti sofisticirane, višestupanjske radne tokove agenata koji mogu rukovati složenim poslovnim procesima i besprijekorno koordinirati više AI operacija.

> **Napomena o migraciji:** Ovaj primjer je prethodno referencirao GitHub modele. GitHub modeli su zastarjeli (ukidaju se u srpnju 2026.), pa sada koriste **Microsoft Foundry** preko `FoundryChatClient`, koji cilja Azure OpenAI **Responses API**.

## 🎯 Ciljevi učenja

### 🏗️ **Arhitektura radnog toka**
- **Workflow Builder**: Dizajnirajte i orkestrirajte složene višestupanjske procese
- **Izvršenje vođeno događajima**: Obradite događaje radnog toka i prijelaze stanja
- **Vizualni dizajn radnog toka**: Kreirajte i vizualizirajte strukture radnog toka
- **Integracija Microsoft Foundryja**: Iskoristite AI modele unutar konteksta radnog toka

### 🔄 **Orkestracija procesa**
- **Sekvencijalne operacije**: Povežite više zadataka agenata u logičkom redoslijedu
- **Uvjetna logika**: Implementirajte točke odluke i razgranate radne tokove
- **Rukovanje pogreškama**: Robusno oporavljanje od pogrešaka i otpornost radnog toka
- **Upravljanje stanjima**: Praćenje i upravljanje stanjem izvršenja radnog toka

### 📊 **Obrasci radnih tokova za poduzeća**
- **Automatizacija poslovnih procesa**: Automatizirajte složene organizacijske radne tokove
- **Koordinacija više agenata**: Koordinirajte više specijaliziranih agenata
- **Izvedba na razini poduzeća**: Dizajnirajte radne tokove za operacije poduzeća u velikom obujmu
- **Nadzor i promatranje**: Pratite izvedbu i ishode radnog toka

## ⚙️ Preduvjeti i postavljanje

### 📦 **Potrebne ovisnosti**

Instalirajte Agent Framework s mogućnostima radnog toka:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundryja**

Prijavite se pomoću Azure CLI-ja (`az login`) kako bi se `AzureCliCredential` mogao autentificirati, zatim postavite detalje vašeg Microsoft Foundry projekta.

**Postavljanje okruženja (datoteka .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Slučajevi upotrebe u poduzećima**

**Primjeri poslovnih procesa:**
- **Uvođenje kupaca**: Višestupanjski tokovi provjere i postavljanja
- **Protokol sadržaja**: Automatizirano stvaranje, pregled i objavljivanje sadržaja
- **Obrada podataka**: ETL radni tokovi s AI-pokretanim transformacijama
- **Osiguranje kvalitete**: Automatizirani procesi testiranja i validacije

**Prednosti radnog toka:**
- 🎯 **Pouzdanost**: Determinističko izvršenje s oporavkom od pogrešaka
- 📈 **Skalabilnost**: Rukovanje automatizacijom procesa visokog volumena
- 🔍 **Promatranje**: Potpuni auditori tragovi i nadzor
- 🔧 **Održavanje**: Vizualni dizajn i modularne komponente

## 🎨 Obrasci dizajna radnih tokova

### Osnovna struktura radnog toka
```mermaid
graph TD
    A[Početak] --> B[Zadatak Agenta 1]
    B --> C{Točka Odluke}
    C -->|Uspjeh| D[Zadatak Agenta 2]
    C -->|Neuspjeh| E[Rukovatelj Pogreškama]
    D --> F[Kraj]
    E --> F
```

**Ključne komponente:**
- **WorkflowBuilder**: Glavni mehanizam orkestracije
- **WorkflowEvent**: Obrada događaja i komunikacija
- **WorkflowViz**: Vizualna reprezentacija i otklanjanje pogrešaka radnog toka

Izgradimo vaš prvi inteligentni radni tok! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
